# YOLOv7 Inference (PyTorch)
Inference simple con modelo YOLOv7 entrenado (.pt)

In [ ]:
import sys
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt

%cd /app/yolov7

from models.experimental import attempt_load
from utils.datasets import LoadImages
from utils.general import check_img_size, non_max_suppression, scale_coords
from utils.plots import plot_one_box
from utils.torch_utils import select_device

In [ ]:
# Configuración básica
weights = '/app/weights/best.pt'  # Ruta al modelo PT
img_size = 1024
conf_thres = 0.10
iou_thres = 0.45

# Configurar dispositivo y modelo
device = select_device('0' if torch.cuda.is_available() else 'cpu')
model = attempt_load(weights, map_location=device)
stride = int(model.stride.max())
img_size = check_img_size(img_size, s=stride)
names = model.module.names if hasattr(model, 'module') else model.names
half = device.type != 'cpu'
if half:
    model.half()

print(f"Dispositivo: {device}")
print(f"Clases del modelo: {names}")

In [ ]:
# Ruta a la imagen de prueba
image_path = "/app/pfs/eosinofilos/dataset/images/test/0001.jpg"

if not os.path.exists(image_path):
    base_dir = "/app/pfs/eosinofilos/dataset/images/test"
    if os.path.exists(base_dir):
        images = list(Path(base_dir).glob("*.jpg")) + list(Path(base_dir).glob("*.png"))
        if images:
            image_path = str(images[0])

# Cargar y preprocesar imagen
dataset = LoadImages(image_path, img_size=img_size, stride=stride)
path, img, im0s, _ = next(iter(dataset))

# Preparar imagen para inferencia
img = torch.from_numpy(img).to(device)
img = img.half() if half else img.float()
img /= 255.0
if img.ndimension() == 3:
    img = img.unsqueeze(0)

# Inferencia
with torch.no_grad():
    pred = model(img, augment=False)[0]

# NMS
pred = non_max_suppression(pred, conf_thres, iou_thres, classes=None, agnostic=False)

# Procesar detecciones
det = pred[0]
im0 = im0s.copy()

if len(det):
    # Reescalar coordenadas
    det[:, :4] = scale_coords(img.shape[2:], det[:, :4], im0.shape).round()
    
    # Mostrar resultados
    print(f"\nDetecciones encontradas: {len(det)}")
    print("-"*50)
    print(f"{'Clase':<15} {'Confianza':<10} {'Coordenadas':<30}")
    print("-"*50)
    
    # Dibujar detecciones
    for *xyxy, conf, cls in reversed(det):
        label = f"{names[int(cls)]} {conf:.2f}"
        plot_one_box(xyxy, im0, label=label, color=(0, 255, 0), line_thickness=2)
        print(f"{names[int(cls)]:<15} {conf:.4f}     {[int(x) for x in xyxy]}")

    # Mostrar imagen
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(im0, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f"Detecciones: {len(det)}")
    plt.show()
else:
    print("No se detectaron objetos en esta imagen.")